In [1]:
# 05_sensitivity.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 05. Sensitivity Analysis for the RCFS
#
# Test whether the headline finding survives perturbations of the RCFS design choices:
#   (H1) a fairness-mitigation technique (reweighing / reject_option) achieves a higher
#        RCFS than the unmitigated baseline, on each dataset
#
# Four perturbation studies:
#   A. EXPL_KSTAR sweep     — vary the explainability target size k*
#   B. Harmonic weights      — reweight c_fair vs c_expl in the compliance term
#   C. Gamma sweep x ranking — does the best technique change with gamma?
#   D. Bootstrap             — resample the test set for RCFS confidence intervals
#
# Component scores are recomputed as in notebook 04 (baseline / reweighing / reject_option).

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
N_BOOTSTRAP = 200
os.makedirs(TAB_DIR, exist_ok=True)

# Baseline RCFS settings (identical to notebook 04)
BASE_KSTAR = 10
GAMMA_DEFAULT = 0.5
W_FAIR, W_EXPL = 1.0, 1.0

DATASETS = [
    ("German Credit",        "german",               "sex", "male"),
    ("Taiwan CC Default",    "taiwan",                "sex", "male"),
    ("Credit Card Approval", "credit_card_approval",  "sex", "male"),
]
TECHNIQUES = ["baseline", "reweighing", "reject_option"]

# %%
# Helpers (mirror notebook 04)
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

def di_spd(y_pred, priv_mask, unpriv_mask):
    ppr_p = y_pred[priv_mask].mean() if priv_mask.sum() else np.nan
    ppr_u = y_pred[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    if not (np.isfinite(ppr_p) and np.isfinite(ppr_u)) or ppr_p == 0:
        return np.nan, np.nan
    return ppr_u / ppr_p, ppr_u - ppr_p

def c_fair_score(test_df, sens_col, privileged, y_pred):
    s = test_df[sens_col]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values
    di, spd = di_spd(y_pred, priv_mask, unpriv_mask)
    if not np.isfinite(di):
        return np.nan
    return 0.5 * min(1.0, di) + 0.5 * (1.0 - abs(spd))

def shap_entropy_neff(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    # Return effective number of features (exp of SHAP-distribution entropy)
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)
    except Exception:
        explainer = shap.Explainer(model, X)
        sv = explainer(X).values
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0:
        return np.nan
    p = mean_abs / total
    p = p[p > 0]
    if len(p) <= 1:
        return 1.0
    entropy = -np.sum(p * np.log(p))
    return float(np.exp(entropy))

def c_expl_from_neff(n_eff, k_star):
    if not np.isfinite(n_eff):
        return np.nan
    return float(min(1.0, k_star / n_eff))

def prevalence_threshold(proba, y_train):
    return np.quantile(proba, 1.0 - float(y_train.mean()))

def group_reweighing_weights(y, s):
    w = np.ones(len(y), dtype=float); n = len(y)
    for yy in np.unique(y):
        for ss in pd.unique(s):
            mask = (y == yy) & (s.values == ss)
            if mask.sum() == 0: continue
            w[mask] = ((y == yy).mean() * (s.values == ss).mean()) / (mask.sum() / n)
    return w

def reject_option_thresholds(proba, s, rate):
    thr = {}
    for grp in pd.unique(s):
        p = proba[(s.values == grp)]
        thr[grp] = np.quantile(p, 1.0 - rate) if len(p) else 0.5
    return thr

def run_technique(technique, X_train, y_train, X_test, s_train, s_test):
    n_pos = int(y_train.sum()); n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    base = dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                subsample=0.9, colsample_bytree=0.9,
                eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    if technique == "baseline":
        model = XGBClassifier(scale_pos_weight=spw, **base); model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        y_pred = (proba >= prevalence_threshold(proba, y_train)).astype(int)
    elif technique == "reweighing":
        w = group_reweighing_weights(y_train.values, s_train)
        model = XGBClassifier(scale_pos_weight=spw, **base); model.fit(X_train, y_train, sample_weight=w)
        proba = model.predict_proba(X_test)[:, 1]
        y_pred = (proba >= prevalence_threshold(proba, y_train)).astype(int)
    elif technique == "reject_option":
        model = XGBClassifier(scale_pos_weight=spw, **base); model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        tm = reject_option_thresholds(proba, s_test, float(y_train.mean()))
        y_pred = np.array([int(p >= tm.get(g, 0.5)) for p, g in zip(proba, s_test.values)])
    return model, proba, y_pred

def compliance_harmonic(cs, ws):
    pairs = [(c, w) for c, w in zip(cs, ws) if np.isfinite(c) and c > 0]
    if not pairs: return np.nan
    return sum(w for _, w in pairs) / sum(w / c for c, w in pairs)

def rcfs(c_perf, compliance, gamma):
    if not np.isfinite(compliance): return np.nan
    return (1 - gamma) * c_perf + gamma * compliance

# %%
# Recompute base component scores once per (dataset, technique) and cache the pieces
# needed to recompute RCFS under different k*, weights, and gamma without retraining.
base_cache = {}   # (dataset, tech) -> dict(c_perf, c_fair, n_eff, y_test, proba, y_pred, s_test, y_train_rate)

for display_name, key, sens_col, privileged in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]
    s_train = train[sens_col]; s_test = test[sens_col]

    for tech in TECHNIQUES:
        model, proba, y_pred = run_technique(tech, X_train, y_train, X_test, s_train, s_test)
        auc = roc_auc_score(y_test.values, proba)
        c_perf = max(0.0, (auc - 0.5) / 0.5)
        c_fair = c_fair_score(test, sens_col, privileged, y_pred)
        n_eff = shap_entropy_neff(model, X_test)
        base_cache[(display_name, tech)] = {
            "c_perf": c_perf, "c_fair": c_fair, "n_eff": n_eff,
            "y_test": y_test.values, "proba": proba, "s_test": s_test,
            "sens_col": sens_col, "privileged": privileged,
            "y_train_rate": float(y_train.mean()), "test_df": test,
        }
    print(f"[{display_name}] base components cached")

# Helper: for a given (k*, w_fair, w_expl, gamma), does a mitigation technique beat baseline?
def best_and_beats_baseline(dataset, k_star, w_fair, w_expl, gamma):
    scores = {}
    for tech in TECHNIQUES:
        c = base_cache[(dataset, tech)]
        c_expl = c_expl_from_neff(c["n_eff"], k_star)
        comp = compliance_harmonic([c["c_fair"], c_expl], [w_fair, w_expl])
        scores[tech] = rcfs(c["c_perf"], comp, gamma)
    best = max(scores, key=scores.get)
    beats = scores[best] > scores["baseline"] + 1e-9  # some mitigation beats baseline
    return best, beats, scores

# %%
# --- Study A: EXPL_KSTAR sweep ---
kstars = [4, 6, 8, 10, 12, 15, 20]
rowsA = []
for ks in kstars:
    for dataset, *_ in DATASETS:
        best, beats, sc = best_and_beats_baseline(dataset, ks, W_FAIR, W_EXPL, GAMMA_DEFAULT)
        rowsA.append({"k_star": ks, "dataset": dataset, "best": best,
                      "beats_baseline": beats, **{f"RCFS_{t}": round(sc[t], 4) for t in TECHNIQUES}})
studyA = pd.DataFrame(rowsA)
print("Study A — EXPL_KSTAR sweep")
print(studyA.to_string(index=False))
studyA.to_csv(os.path.join(TAB_DIR, "05_sensitivity_kstar.csv"), index=False)
print("H1 holds across all k*:", studyA["beats_baseline"].all())

# %%
# --- Study B: harmonic weights (c_fair vs c_expl) ---
weight_grid = [(2.0, 1.0), (1.5, 1.0), (1.0, 1.0), (1.0, 1.5), (1.0, 2.0)]
rowsB = []
for wf, we in weight_grid:
    for dataset, *_ in DATASETS:
        best, beats, sc = best_and_beats_baseline(dataset, BASE_KSTAR, wf, we, GAMMA_DEFAULT)
        rowsB.append({"w_fair": wf, "w_expl": we, "dataset": dataset,
                      "best": best, "beats_baseline": beats})
studyB = pd.DataFrame(rowsB)
print("Study B — harmonic weights")
print(studyB.to_string(index=False))
studyB.to_csv(os.path.join(TAB_DIR, "05_sensitivity_weights.csv"), index=False)
print("H1 holds across all weightings:", studyB["beats_baseline"].all())

# %%
# --- Study C: gamma sweep and best-technique stability ---
gammas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
rowsC = []
for g in gammas:
    for dataset, *_ in DATASETS:
        best, beats, sc = best_and_beats_baseline(dataset, BASE_KSTAR, W_FAIR, W_EXPL, g)
        rowsC.append({"gamma": g, "dataset": dataset, "best": best, "beats_baseline": beats})
studyC = pd.DataFrame(rowsC)
print("Study C — gamma sweep")
print(studyC.pivot(index="gamma", columns="dataset", values="best").to_string())
studyC.to_csv(os.path.join(TAB_DIR, "05_sensitivity_gamma.csv"), index=False)
print("H1 holds across all gamma (gamma>0):",
      studyC[studyC["gamma"] > 0]["beats_baseline"].all())

# %%
# --- Study D: bootstrap 95% CI for RCFS per (dataset, technique) at baseline settings ---
def bootstrap_rcfs(dataset, tech, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    c = base_cache[(dataset, tech)]
    rng = np.random.default_rng(seed)
    y = c["y_test"]; proba = c["proba"]; s = c["s_test"].reset_index(drop=True)
    test_df = c["test_df"].reset_index(drop=True)
    n = len(y)
    # explainability is held fixed (SHAP per replicate is costly)
    c_expl = c_expl_from_neff(c["n_eff"], BASE_KSTAR)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb = y[idx]
        if len(np.unique(yb)) < 2:
            continue
        pb = proba[idx]
        auc = roc_auc_score(yb, pb)
        c_perf = max(0.0, (auc - 0.5) / 0.5)
        # recompute predictions/fairness on the resample (technique-consistent threshold)
        if tech == "reject_option":
            tm = reject_option_thresholds(pb, s.iloc[idx], c["y_train_rate"])
            yhat = np.array([int(p >= tm.get(g, 0.5)) for p, g in zip(pb, s.iloc[idx].values)])
        else:
            thr = np.quantile(pb, 1.0 - c["y_train_rate"])
            yhat = (pb >= thr).astype(int)
        c_fair = c_fair_score(test_df.iloc[idx], c["sens_col"], c["privileged"], yhat)
        comp = compliance_harmonic([c_fair, c_expl], [W_FAIR, W_EXPL])
        vals.append(rcfs(c_perf, comp, GAMMA_DEFAULT))
    arr = np.array([v for v in vals if np.isfinite(v)])
    return arr

rowsD = []
for dataset, *_ in DATASETS:
    for tech in TECHNIQUES:
        arr = bootstrap_rcfs(dataset, tech)
        rowsD.append({
            "dataset": dataset, "technique": tech,
            "RCFS_mean": round(arr.mean(), 4),
            "CI_low": round(np.percentile(arr, 2.5), 4),
            "CI_high": round(np.percentile(arr, 97.5), 4),
            "n_valid": len(arr),
        })
studyD = pd.DataFrame(rowsD)
print("Study D — bootstrap 95% CI")
print(studyD.to_string(index=False))
studyD.to_csv(os.path.join(TAB_DIR, "05_sensitivity_bootstrap.csv"), index=False)

# %%
# --- Overall verdict ---
print("=== Robustness summary ===")
print("H1 (a mitigation beats baseline) holds across k*:", studyA["beats_baseline"].all())
print("H1 holds across harmonic weights:", studyB["beats_baseline"].all())
print("H1 holds across gamma>0:", studyC[studyC["gamma"] > 0]["beats_baseline"].all())

[German Credit] base components cached
[Taiwan CC Default] base components cached
[Credit Card Approval] base components cached
Study A — EXPL_KSTAR sweep
 k_star              dataset          best  beats_baseline  RCFS_baseline  RCFS_reweighing  RCFS_reject_option
      4        German Credit    reweighing            True         0.4192           0.4319              0.4194
      4    Taiwan CC Default reject_option            True         0.4789           0.4789              0.4823
      4 Credit Card Approval reject_option            True         0.3466           0.3504              0.3507
      6        German Credit    reweighing            True         0.4680           0.4812              0.4684
      6    Taiwan CC Default reject_option            True         0.5507           0.5507              0.5569
      6 Credit Card Approval reject_option            True         0.4051           0.4122              0.4129
      8        German Credit    reweighing            True         0